# State Farm VGG16: blocks 4–5 미세조정

57.03%의 기준 모델 체크포인트에서 이어 학습합니다. 분류기와 VGG16의 마지막
두 convolution 블록을 학습하며, 초기 세 블록은 고정합니다.
운전자별 검증 분할(seed=42)은 기준 모델과 완전히 같습니다.

* 최대 12에폭, 검증 정확도 조기 종료 patience=4, cosine 학습률 감소.
* backbone LR=3e-5, classifier LR=3e-4, AdamW, label smoothing=0.05.
* 224×224, 약한 색상·기하 증강. 좌우 행동 구분을 위해 좌우 반전은 사용하지 않습니다.
* 두 GPU가 있으면 DataParallel로 사용하며, batch size=64입니다.
* 검증 손실은 label smoothing 없는 multiclass Log Loss입니다.
* 정확도 기준과 Log Loss 기준의 최적 체크포인트를 각각 저장합니다.
* 공식 대회 test 이미지는 학습과 검증에 사용하지 않습니다.
* 이 검증 분할은 모델 선택에도 사용되므로 독립적인 최종 테스트 점수는 아닙니다.


In [ ]:
import gc
import hashlib
import json
import os
import random
import re
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm

SEED = 42
EPOCHS = 12
PATIENCE = 4
BATCH_SIZE = 64
NUM_WORKERS = 4
BACKBONE_LR = 3e-5
HEAD_LR = 3e-4
WEIGHT_DECAY = 1e-4
CLASS_NAMES = [f'c{i}' for i in range(10)]
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/vgg16_finetune')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError('GPU가 필요합니다. Kaggle Settings → Accelerator에서 GPU를 선택하세요.')
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda:0')
print({'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0),
       'seed': SEED, 'epochs': EPOCHS})

print({'available_gpus': torch.cuda.device_count(), 'max_epochs': EPOCHS}, flush=True)


## 공식 메타데이터와 이미지 찾기
압축 해제된 이미지와 ZIP 형태 모두 지원합니다.
ZIP 이미지는 필요한 항목만 읽으며, 라벨 없는 test 이미지를 압축 해제하지 않습니다.


In [ ]:
def input_files(root, pattern):
    # Kaggle mounts datasets directly or under /input/competitions/<slug>.
    # Avoid recursively listing the competition's 79k unlabelled test images.
    paths = set()
    for prefix in ('', '*/', '*/*/'):
        paths.update(root.glob(prefix + pattern))
    return sorted(paths)

def read_driver_metadata(root):
    csv_paths = input_files(root, 'driver_imgs_list.csv')
    if len(csv_paths) > 1:
        raise RuntimeError(f'메타데이터가 여러 개입니다. 공식 대회 입력만 연결하세요: {csv_paths}')
    if csv_paths:
        return pd.read_csv(csv_paths[0]), str(csv_paths[0])
    candidates = []
    for archive in input_files(root, '*.zip'):
        with zipfile.ZipFile(archive) as zf:
            members = [n for n in zf.namelist() if Path(n).name == 'driver_imgs_list.csv']
            for member in members:
                candidates.append((archive, member))
    if len(candidates) != 1:
        raise FileNotFoundError('driver_imgs_list.csv를 하나만 찾을 수 있어야 합니다. 공식 대회 데이터를 연결하세요.')
    archive, member = candidates[0]
    with zipfile.ZipFile(archive) as zf, zf.open(member) as file:
        return pd.read_csv(file), f'{archive}!{member}'

metadata, metadata_source = read_driver_metadata(INPUT_ROOT)
required = {'subject', 'classname', 'img'}
if not required.issubset(metadata.columns):
    raise ValueError(f'필수 열 누락: {required - set(metadata.columns)}')
metadata = metadata[['subject', 'classname', 'img']].copy()
if metadata.isna().any().any():
    raise ValueError('메타데이터에 결측값이 있습니다.')
if set(metadata['classname']) != set(CLASS_NAMES):
    raise ValueError('c0부터 c9까지 10개 클래스가 모두 필요합니다.')
if metadata.duplicated(['classname', 'img']).any() or metadata['img'].duplicated().any():
    raise ValueError('중복 이미지 식별자가 있습니다.')
metadata['label'] = metadata['classname'].str[1:].astype(int)

# Store only locators, never image bytes. Each worker opens its own ZIP handles.
wanted = set(zip(metadata['classname'], metadata['img']))
image_index = {}

def add_image(key, locator):
    if key not in wanted:
        return
    if key in image_index:
        raise RuntimeError(f'이미지 중복 입력: {key}. 같은 데이터를 두 번 연결하지 마세요.')
    image_index[key] = locator

dataset_root = Path(metadata_source.split('!')[0]).parent
train_roots = [p for p in (dataset_root / 'imgs/train', dataset_root / 'train') if p.is_dir()]
for train_root in train_roots:
    for classname in CLASS_NAMES:
        for image_path in (train_root / classname).glob('*.jpg'):
            add_image((classname, image_path.name), ('file', str(image_path), ''))

# Prefer a complete extracted copy. Otherwise supplement from ZIP archives.
if len(image_index) != len(wanted):
    found_on_disk = set(image_index)
    archive_pattern = re.compile(r'(?:^|/)(c[0-9])/([^/]+\.jpg)$', re.IGNORECASE)
    for archive in input_files(INPUT_ROOT, '*.zip'):
        with zipfile.ZipFile(archive) as zf:
            for member in zf.namelist():
                match = archive_pattern.search(member.replace('\\', '/'))
                if match:
                    key = (match.group(1).lower(), match.group(2))
                    if key not in found_on_disk:
                        add_image(key, ('zip', str(archive), member))
missing = wanted - set(image_index)
if missing:
    raise FileNotFoundError(f'{len(missing)}개 학습 이미지 누락. 예시: {sorted(missing)[:5]}')
metadata_sha256 = hashlib.sha256(metadata.to_csv(index=False).encode()).hexdigest()
print({'metadata': metadata_source, 'rows': len(metadata),
       'drivers': metadata['subject'].nunique(), 'images_found': len(image_index)})


In [ ]:
def split_by_driver(frame):
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    train_ids, val_ids = next(splitter.split(frame, frame['label'], groups=frame['subject']))
    train = frame.iloc[train_ids].reset_index(drop=True)
    val = frame.iloc[val_ids].reset_index(drop=True)
    if set(train['subject']) & set(val['subject']):
        raise RuntimeError('운전자 누출이 발견되었습니다.')
    if set(train['img']) & set(val['img']):
        raise RuntimeError('이미지 누출이 발견되었습니다.')
    expected = set(range(10))
    if set(train['label']) != expected or set(val['label']) != expected:
        raise RuntimeError('학습과 검증에 각각 10개 클래스가 모두 있어야 합니다.')
    return train, val

train_df, val_df = split_by_driver(metadata)
train_df.to_csv(OUTPUT_ROOT / 'train_split.csv', index=False)
val_df.to_csv(OUTPUT_ROOT / 'validation_split.csv', index=False)
split_report = {
    'method': 'GroupShuffleSplit(subject)', 'seed': SEED,
    'train_images': len(train_df), 'validation_images': len(val_df),
    'train_drivers': sorted(train_df['subject'].unique().tolist()),
    'validation_drivers': sorted(val_df['subject'].unique().tolist()),
    'class_order': CLASS_NAMES, 'metadata_sha256': metadata_sha256,
}
(OUTPUT_ROOT / 'split_report.json').write_text(json.dumps(split_report, indent=2))
print(json.dumps(split_report, indent=2))
display(pd.DataFrame({
    'train': train_df['classname'].value_counts(),
    'validation': val_df['classname'].value_counts(),
}).reindex(CLASS_NAMES))


In [ ]:
class DriverImages(Dataset):
    def __init__(self, frame, image_index, transform):
        self.frame = frame.reset_index(drop=True)
        self.image_index = image_index
        self.transform = transform
        self._archives = {}
        self._pid = None

    def __len__(self):
        return len(self.frame)

    def __getstate__(self):
        state = self.__dict__.copy()
        state['_archives'] = {}
        state['_pid'] = None
        return state

    def close(self):
        for archive in self._archives.values():
            archive.close()
        self._archives = {}

    def __getitem__(self, index):
        if self._pid != os.getpid():
            self.close()
            self._pid = os.getpid()
        row = self.frame.iloc[index]
        kind, path, member = self.image_index[(row['classname'], row['img'])]
        if kind == 'file':
            with Image.open(path) as image:
                image = image.convert('RGB')
        else:
            if path not in self._archives:
                self._archives[path] = zipfile.ZipFile(path)
            with self._archives[path].open(member) as file:
                with Image.open(file) as image:
                    image = image.convert('RGB')
        return self.transform(image), int(row['label'])

normalization = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomAffine(degrees=7, translate=(0.04, 0.04), scale=(0.95, 1.05), fill=(123,116,103)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(), normalization,
])
validation_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(), normalization,
])
train_dataset = DriverImages(train_df, image_index, train_transform)
val_dataset = DriverImages(val_df, image_index, validation_transform)
# Inspect image decoding before starting workers or downloading weights.
for dataset in (train_dataset, val_dataset):
    x, y = dataset[0]
    assert tuple(x.shape) == (3, 224, 224) and 0 <= y < 10
    dataset.close()

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True,
                   worker_init_fn=seed_worker, persistent_workers=NUM_WORKERS > 0)
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, shuffle=True, generator=generator, **loader_args)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_args)


In [ ]:
checkpoint_paths = set()
for pattern in ('*/vgg16_baseline/vgg16_baseline.pt',
                '*/*/vgg16_baseline/vgg16_baseline.pt',
                'notebooks/*/*/vgg16_baseline/vgg16_baseline.pt'):
    checkpoint_paths.update(INPUT_ROOT.glob(pattern))
if len(checkpoint_paths) != 1:
    raise FileNotFoundError(f'기준 모델 입력이 하나 필요합니다: {sorted(checkpoint_paths)}')
baseline_checkpoint_path = next(iter(checkpoint_paths))
checkpoint = torch.load(baseline_checkpoint_path, map_location='cpu', weights_only=True)
if checkpoint['split_report']['validation_drivers'] != split_report['validation_drivers']:
    raise RuntimeError('기준 모델과 검증 운전자가 다릅니다.')
if checkpoint['metrics']['metadata_sha256'] != metadata_sha256:
    raise RuntimeError('기준 모델과 데이터 메타데이터가 다릅니다.')

class VGG16FineTune(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.features = features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(512, 10))

    def forward(self, x):
        # Do not wrap trainable features in no_grad. Frozen layers have no grad graph automatically.
        x = self.pool(self.features(x)).flatten(1)
        return self.classifier(x)

original_vgg = models.vgg16(weights=None)
core = VGG16FineTune(original_vgg.features)
del original_vgg
core.load_state_dict(checkpoint['model_state_dict'], strict=True)
baseline_metrics = checkpoint['metrics']
del checkpoint
core.features.requires_grad_(True)
core.features[:17].requires_grad_(False)  # Freeze VGG blocks 1–3, fine-tune blocks 4–5.
core = core.to(device)
model = nn.DataParallel(core) if torch.cuda.device_count() > 1 else core
gc.collect()
torch.cuda.empty_cache()

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
validation_criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW([
    {'params': [p for p in core.features.parameters() if p.requires_grad], 'lr': BACKBONE_LR},
    {'params': core.classifier.parameters(), 'lr': HEAD_LR},
], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')
trainable_parameters = sum(p.numel() for p in core.parameters() if p.requires_grad)
print({'baseline_checkpoint': str(baseline_checkpoint_path),
       'trainable_parameters': trainable_parameters,
       'gpu_count_used': torch.cuda.device_count(),
       'backbone_lr': BACKBONE_LR, 'head_lr': HEAD_LR,
       'frozen_blocks': [1, 2, 3], 'trainable_blocks': [4, 5]}, flush=True)


In [ ]:
def evaluate(model, loader):
    model.eval()
    targets, probabilities = [], []
    loss_sum = 0.0
    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(images)
            loss_sum += validation_criterion(logits.float(), labels).item() * len(labels)
            probabilities.append(logits.double().softmax(dim=1).cpu().numpy())
            targets.append(labels.cpu().numpy())
    targets = np.concatenate(targets)
    probabilities = np.concatenate(probabilities)
    if not np.isfinite(probabilities).all():
        raise RuntimeError('검증 확률이 유한하지 않습니다.')
    np.testing.assert_allclose(probabilities.sum(axis=1), 1, atol=1e-8)
    accuracy = float(accuracy_score(targets, probabilities.argmax(axis=1)))
    score = float(log_loss(targets, probabilities, labels=list(range(10))))
    if not np.isclose(loss_sum / len(targets), score, rtol=1e-5, atol=1e-5):
        raise RuntimeError('검증 CE와 Log Loss가 일치하지 않습니다.')
    return targets, probabilities, accuracy, score

def prediction_frame(probabilities):
    result = val_df[['subject', 'classname', 'img', 'label']].copy()
    for i, name in enumerate(CLASS_NAMES):
        result[name] = probabilities[:, i]
    return result

def save_checkpoint(filename, row):
    state = {name: value.detach().cpu() for name, value in core.state_dict().items()}
    torch.save({'model_state_dict': state, 'metrics': row, 'split_report': split_report,
                'architecture': 'VGG16-GAP-Dropout0.3-Linear512x10',
                'frozen_blocks': [1, 2, 3]}, OUTPUT_ROOT / filename)

print('Checking resumed baseline on the unchanged validation split...', flush=True)
targets, initial_prob, initial_accuracy, initial_log_loss = evaluate(model, val_loader)
if abs(initial_accuracy - baseline_metrics['validation_accuracy']) > 0.001:
    raise RuntimeError('재개한 모델의 정확도가 기준 모델과 일치하지 않습니다.')
if abs(initial_log_loss - baseline_metrics['validation_log_loss']) > 0.005:
    raise RuntimeError('재개한 모델의 Log Loss가 기준 모델과 일치하지 않습니다.')
initial = {'epoch': 0, 'validation_accuracy': initial_accuracy,
           'validation_log_loss': initial_log_loss}
print(json.dumps({'resumed_baseline': initial}), flush=True)
best_accuracy_row = dict(initial)
best_log_loss_row = dict(initial)
save_checkpoint('best_accuracy.pt', initial)
save_checkpoint('best_log_loss.pt', initial)
prediction_frame(initial_prob).to_csv(OUTPUT_ROOT / 'validation_predictions.csv', index=False)
prediction_frame(initial_prob).to_csv(OUTPUT_ROOT / 'best_log_loss_predictions.csv', index=False)
history = []
stale_epochs = 0
started = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    epoch_started = time.perf_counter()
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_seen = 0
    current_lrs = [group['lr'] for group in optimizer.param_groups]
    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(images)
        loss = criterion(logits.float(), labels)
        if not torch.isfinite(loss):
            raise RuntimeError('학습 손실이 유한하지 않습니다.')
        scaler.scale(loss).backward()
        if epoch == 1 and batch_idx == 0:
            assert core.features[28].weight.grad is not None, '미세조정 계층에 gradient가 없습니다.'
            assert core.features[0].weight.grad is None, '고정 계층에 gradient가 있습니다.'
            print('Gradient check passed: blocks 4–5 update; blocks 1–3 stay frozen.', flush=True)
        scaler.step(optimizer)
        scaler.update()
        train_loss_sum += loss.item() * len(labels)
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_seen += len(labels)
        if (batch_idx + 1) % 100 == 0:
            print(json.dumps({'epoch': epoch, 'batches_completed': batch_idx + 1,
                              'batches_total': len(train_loader),
                              'running_train_accuracy': train_correct / train_seen}), flush=True)
    targets, probabilities, val_accuracy, val_log_loss = evaluate(model, val_loader)
    np.testing.assert_array_equal(targets, val_df['label'].to_numpy())
    row = {'epoch': epoch, 'train_loss': train_loss_sum / train_seen,
           'train_accuracy': train_correct / train_seen,
           'validation_accuracy': val_accuracy, 'validation_log_loss': val_log_loss,
           'backbone_lr': current_lrs[0], 'head_lr': current_lrs[1],
           'epoch_seconds': time.perf_counter() - epoch_started}
    history.append(row)
    pd.DataFrame(history).to_csv(OUTPUT_ROOT / 'history.csv', index=False)
    improved_accuracy = (val_accuracy > best_accuracy_row['validation_accuracy'] + 1e-8 or
        (abs(val_accuracy - best_accuracy_row['validation_accuracy']) <= 1e-8 and
         val_log_loss < best_accuracy_row['validation_log_loss']))
    if improved_accuracy:
        best_accuracy_row = dict(row)
        stale_epochs = 0
        save_checkpoint('best_accuracy.pt', row)
        prediction_frame(probabilities).to_csv(OUTPUT_ROOT / 'validation_predictions.csv', index=False)
    else:
        stale_epochs += 1
    if val_log_loss < best_log_loss_row['validation_log_loss']:
        best_log_loss_row = dict(row)
        save_checkpoint('best_log_loss.pt', row)
        prediction_frame(probabilities).to_csv(OUTPUT_ROOT / 'best_log_loss_predictions.csv', index=False)
    print(json.dumps({**row, 'best_accuracy_epoch': best_accuracy_row['epoch'],
                      'best_validation_accuracy': best_accuracy_row['validation_accuracy'],
                      'stale_epochs': stale_epochs}), flush=True)
    scheduler.step()
    if stale_epochs >= PATIENCE:
        print(f'Early stopping after epoch {epoch}.', flush=True)
        break

best = torch.load(OUTPUT_ROOT / 'best_accuracy.pt', map_location='cpu', weights_only=True)
core.load_state_dict(best['model_state_dict'], strict=True)
del best
targets, probabilities, final_accuracy, final_log_loss = evaluate(model, val_loader)
if not np.isclose(final_accuracy, best_accuracy_row['validation_accuracy'], atol=1e-8):
    raise RuntimeError('최적 정확도 체크포인트를 다시 평가한 값이 일치하지 않습니다.')
if not np.isclose(final_log_loss, best_accuracy_row['validation_log_loss'], rtol=1e-5, atol=1e-5):
    raise RuntimeError('최적 체크포인트의 Log Loss가 재평가와 일치하지 않습니다.')
prediction_frame(probabilities).to_csv(OUTPUT_ROOT / 'validation_predictions.csv', index=False)
pd.DataFrame(confusion_matrix(targets, probabilities.argmax(axis=1), labels=list(range(10))),
             index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(OUTPUT_ROOT / 'confusion_matrix.csv')
per_class = []
for label, classname in enumerate(CLASS_NAMES):
    mask = targets == label
    per_class.append({'class': classname, 'count': int(mask.sum()),
                      'accuracy': float((probabilities[mask].argmax(axis=1) == label).mean())})
pd.DataFrame(per_class).to_csv(OUTPUT_ROOT / 'per_class_metrics.csv', index=False)
metrics = {
    'validation_accuracy': final_accuracy, 'validation_log_loss': final_log_loss,
    'best_epoch': best_accuracy_row['epoch'], 'epochs_completed': len(history),
    'max_epochs': EPOCHS, 'baseline_validation_accuracy': initial_accuracy,
    'baseline_validation_log_loss': initial_log_loss,
    'accuracy_improvement_percentage_points': (final_accuracy - initial_accuracy) * 100,
    'lowest_log_loss_checkpoint': best_log_loss_row,
    'gpu': torch.cuda.get_device_name(0), 'gpu_count_used': torch.cuda.device_count(),
    'train_images': len(train_df), 'validation_images': len(val_df),
    'seed': SEED, 'trainable_parameters': trainable_parameters,
    'frozen_blocks': [1, 2, 3], 'trainable_blocks': [4, 5],
    'batch_size': BATCH_SIZE, 'label_smoothing': 0.05,
    'backbone_lr': BACKBONE_LR, 'head_lr': HEAD_LR, 'weight_decay': WEIGHT_DECAY,
    'elapsed_seconds': time.perf_counter() - started,
    'metadata_sha256': metadata_sha256,
    'selection_note': 'Same driver-disjoint holdout used for checkpoint selection; not independent test accuracy.',
}
(OUTPUT_ROOT / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print('FINAL_METRICS ' + json.dumps(metrics), flush=True)


## 결과

`vgg16_finetune/metrics.json`은 최적 정확도 체크포인트를 다시 평가한 결과입니다.
`history.csv`에 모든 에폭 결과가 있고, `best_accuracy.pt` 및 `best_log_loss.pt`를
각각 보존합니다. 동일한 운전자별 holdout에서 기준 모델과 비교하며,
독립적인 최종 테스트 평가는 수행하지 않습니다.
